<a href="https://colab.research.google.com/github/StereoWings7/potential-octo-giggle/blob/main/OCR_dataset_prepare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
import kagglehub
import matplotlib.pyplot as plt
%matplotlib inline
import os

In [ ]:
os.chdir('/content/drive/MyDrive/Colab Notebooks')

In [ ]:
os.getcwd()

'/content/drive/MyDrive/Colab Notebooks'

In [ ]:
#まだダウンロードしてない場合はここを実行
path = kagglehub.dataset_download("harieh/ocr-dataset")

100%|██████████| 153M/153M [00:02<00:00, 79.0MB/s]

Extracting files...


In [ ]:
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/harieh/ocr-dataset/versions/1


In [ ]:
#Googleドライブ側でコピーしておく
data_dir = "/content/drive/MyDrive/data/OCR-dataset"

In [ ]:
#別途計算した平均分散を元に正規化
mean = (0.912,)
std = (0.068,)
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # グレースケール化
    transforms.Resize((96,96)),  # サイズを統一
    transforms.ToTensor(),  # Tensorに変換 (形状: [1, H, W])
    transforms.Normalize(mean,std)
])

In [ ]:
#画像データをImageFolderを使って取込みする
#3分くらいかかる模様
train_dataset = ImageFolder(data_dir, transform = train_transform)

In [ ]:
batch_size=32
#上で作成したdatasetからDataloaderを作る
train_dataloader = DataLoader(
     train_dataset, batch_size=batch_size, shuffle=True
)

In [ ]:
# 1バッチ取得だけ取得して、中身を確認してみる。
images, labels = next(iter(train_dataloader))

print("画像のテンソルの形状:", images.shape)  # 例: torch.Size([4, 3, 64, 64])
print("ラベル:", labels)


画像のテンソルの形状: torch.Size([32, 3, 144, 96])
ラベル: tensor([28, 57, 37, 20, 22, 23, 35, 41, 13, 52,  7, 11,  3, 53, 51,  4,  9, 27,
        29, 25, 25,  0,  1, 52, 58, 22, 25, 40, 50, 17, 10, 42])


In [ ]:
print(images[1][0])

tensor([[1.2941, 1.2941, 1.2941,  ..., 1.2941, 1.2941, 1.2941],
        [1.2941, 1.2941, 1.2941,  ..., 1.2941, 1.2941, 1.2941],
        [1.2941, 1.2941, 1.2941,  ..., 1.2941, 1.2941, 1.2941],
        ...,
        [1.2941, 1.2941, 1.2941,  ..., 1.2941, 1.2941, 1.2941],
        [1.2941, 1.2941, 1.2941,  ..., 1.2941, 1.2941, 1.2941],
        [1.2941, 1.2941, 1.2941,  ..., 1.2941, 1.2941, 1.2941]])


In [ ]:
#正規化はしているが、バッチごとにランダムにピックアップしているので、バッチ内の平均/分散は1にはならない
print(torch.mean(images))
print(torch.std(images))

tensor(-0.2791)
tensor(4.1783)


In [ ]:
# あとはこれまでと同様にモデル作成、学習ループを回す